# 🧪 [Day 30] Cypher 심화(UNWIND·WITH·OPTIONAL MATCH·다중 홉) 실전 워크북

- **과정 구분**: 지식그래프 엔지니어링 실전 마스터
- **데이터셋**: [DART-Trace] 3,746건 기업공시 지분 데이터 & [ART:READY] 중앙대·한양대 미대입시 실전 요강
- **핵심 미션**: UNWIND 대량 배치 적재, OPTIONAL MATCH 결손 일정 방어, WITH 체이닝, 다중 홉 지분 탐색을 직접 실습한다.

## 1. 환경 설정 및 Neo4j 드라이버 연결

In [ ]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
print("✅ Neo4j 연결 성공:", NEO4J_URI)

## 2. [DART-Trace] UNWIND 배치 적재 실습
파이썬에서 수십 개의 주주 지분 리스트를 단 1회의 Cypher 호출로 언패킹하여 대량 적재합니다.

In [ ]:
dart_batch = [
    {"corp_code": "00126380", "corp_name": "삼성전자", "market_type": "KOSPI", "holder_key": "HOLDER_NPS", "holder_name": "국민연금공단", "holder_type": "기관투자자", "stake_ratio": 7.25, "base_date": "2024-03-31"},
    {"corp_code": "00126380", "corp_name": "삼성전자", "market_type": "KOSPI", "holder_key": "HOLDER_SAMSUNG_LIFE", "holder_name": "삼성생명보험", "holder_type": "계열회사", "stake_ratio": 8.51, "base_date": "2024-03-31"},
    {"corp_code": "000660", "corp_name": "SK하이닉스", "market_type": "KOSPI", "holder_key": "HOLDER_SK_SQUARE", "holder_name": "SK스퀘어", "holder_type": "최대주주", "stake_ratio": 20.07, "base_date": "2024-03-31"},
    {"corp_code": "000660", "corp_name": "SK하이닉스", "market_type": "KOSPI", "holder_key": "HOLDER_NPS", "holder_name": "국민연금공단", "holder_type": "기관투자자", "stake_ratio": 7.90, "base_date": "2024-03-31"}
]

unwind_query = """
UNWIND $batch AS row
MERGE (c:Company {corp_code: row.corp_code})
  ON CREATE SET c.name = row.corp_name, c.market_type = row.market_type
MERGE (s:Shareholder {holder_key: row.holder_key})
  ON CREATE SET s.name = row.holder_name, s.holder_type = row.holder_type
MERGE (s)-[r:HOLDS_ECONOMIC_STAKE]->(c)
  ON CREATE SET r.stake_ratio = row.stake_ratio, r.base_date = row.base_date
RETURN count(r) AS loaded_count;
"""

with driver.session() as session:
    res = session.run(unwind_query, batch=dart_batch).single()
    print(f"✅ UNWIND 배치 적재 완료: 총 {res['loaded_count']}개 관계 생성/갱신")

## 3. [ART:READY] OPTIONAL MATCH 결손 일정 방어 조회
고사 일정이 아직 확정되지 않은 대학 전형도 누락 없이 `coalesce`와 함께 안전하게 표시합니다.

In [ ]:
opt_match_query = """
MATCH (u:University)-[:OFFERS_TRACK]->(t:AdmissionTrack)-[r:REQUIRES_PRACTICAL]->(p:PracticalType)
OPTIONAL MATCH (t)-[:EXAM_ON]->(e:ExamSchedule)
RETURN 
    u.name AS university,
    u.campus AS campus,
    t.name AS track_name,
    p.name AS subject,
    r.ratio AS practical_ratio,
    coalesce(e.exam_date, '일정 미발표(TBD)') AS exam_date
ORDER BY u.name, t.name;
"""

with driver.session() as session:
    records = list(session.run(opt_match_query))
    print(f"✅ 총 {len(records)}개 전형 조회 성공")
    for r in records:
        print(f"[{r['university']}] {r['track_name']} | 실기: {r['subject']}({r['practical_ratio']}%) | 고사일: {r['exam_date']}")

## 4. WITH 절 체이닝 및 다중 홉(Multi-Hop) 지분 네트워크 탐색
중간 집계 후 2개 이상 기업에 동시 출자한 큰손 기관투자자만 필터링합니다.

In [ ]:
with_query = """
MATCH (s:Shareholder)-[r:HOLDS_ECONOMIC_STAKE]->(c:Company)
WITH s, count(c) AS cnt, collect(c.name) AS corps, avg(r.stake_ratio) AS avg_stake
WHERE cnt >= 2
RETURN s.name AS shareholder, cnt, corps, round(avg_stake, 2) AS avg_stake;
"""

with driver.session() as session:
    records = list(session.run(with_query))
    for r in records:
        print(f"• {r['shareholder']}: {r['cnt']}개사 투자 {r['corps']} (평균 지분율: {r['avg_stake']}%)")